# 09 - Ring-based spillover estimation

This notebook estimates final ring-based spillover diagnostics using untreated cells. It reports HT and Hajek contrasts by year and ring, relative to cells farther than the reference distance from treated cells.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'notebooks').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.estimation_workflow_utils import append_tag_to_filename, ht_hajek_ring_diagnostics, hac_for_hajek_ring_contrasts

DATA_DIR = PROJECT_ROOT / 'data'
INTERMEDIATE_DIR = DATA_DIR / 'intermediate'
SPATIAL_DIR = INTERMEDIATE_DIR / 'spatial_structure'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
TABLE_DIR = OUTPUT_DIR / 'tables'
FIG_DIR = OUTPUT_DIR / 'figures'
for path in [TABLE_DIR, FIG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)


## User configuration

In [ ]:
RUN_TAG = '1km'
START_YEAR = 2001
END_YEAR = None
TREATMENT_KEY = 'protected_area'
CELL_COL = 'cell_id'
YEAR_COL = 'year'
OUTCOME_COL = 'loss_m2'
REFERENCE_DISTANCE_KM = 50

PANEL_PATH = INTERMEDIATE_DIR / f'panel_treatment_{RUN_TAG}.parquet'
if not PANEL_PATH.exists():
    PANEL_PATH = INTERMEDIATE_DIR / 'panel_treatment.parquet'
EXPOSURE_PATH = INTERMEDIATE_DIR / append_tag_to_filename('06_nearest_treated_exposure.parquet', RUN_TAG)
CENTROID_PATH = SPATIAL_DIR / append_tag_to_filename('03_grid_centroids.parquet', RUN_TAG)
if not CENTROID_PATH.exists():
    CENTROID_PATH = SPATIAL_DIR / '03_grid_centroids.parquet'

SPILLOVER_TABLE_NAME = '09_ring_spillover_ht_hajek.csv'
SPILLOVER_EVENT_TABLE_NAME = '09_ring_spillover_event_time.csv'
SPILLOVER_HAC_TABLE_NAME = '09_ring_spillover_spatial_hac.csv'
HAC_CUTOFF_KM = 25
HAC_KERNEL = 'bartlett'
HAC_MAX_PAIRS = 50_000_000
SPILLOVER_FIG_NAME = '09_ring_spillover_profiles.png'


## Load untreated outcome cells and exposure rings

In [ ]:
treated_col = f'treated_it_{TREATMENT_KEY}'
panel_cols = [CELL_COL, YEAR_COL, OUTCOME_COL, treated_col, 'first_treat_year', 'event_time']
try:
    panel = pd.read_parquet(PANEL_PATH, columns=panel_cols)
except Exception:
    treated_col = 'treated_it'
    panel_cols = [CELL_COL, YEAR_COL, OUTCOME_COL, treated_col, 'first_treat_year', 'event_time']
    panel = pd.read_parquet(PANEL_PATH, columns=panel_cols)
panel[CELL_COL] = panel[CELL_COL].astype('string')
panel[YEAR_COL] = pd.to_numeric(panel[YEAR_COL], errors='coerce').astype(int)
panel = panel[panel[YEAR_COL] >= START_YEAR].copy()
if END_YEAR is not None:
    panel = panel[panel[YEAR_COL] <= END_YEAR].copy()

exposure = pd.read_parquet(EXPOSURE_PATH)
exposure[CELL_COL] = exposure[CELL_COL].astype('string')
exposure[YEAR_COL] = pd.to_numeric(exposure[YEAR_COL], errors='coerce').astype(int)
centroids = pd.read_parquet(CENTROID_PATH)
centroids[CELL_COL] = centroids[CELL_COL].astype('string')

df = panel.merge(exposure[[CELL_COL, YEAR_COL, 'nearest_treated_distance_km', 'distance_ring']], on=[CELL_COL, YEAR_COL], how='left')
df = df[pd.to_numeric(df[treated_col], errors='coerce').fillna(0).astype(int) == 0].copy()
df['spillover_ring'] = df['distance_ring'].astype('string')
df.loc[df['nearest_treated_distance_km'] > REFERENCE_DISTANCE_KM, 'spillover_ring'] = f'gt_{REFERENCE_DISTANCE_KM}km'
reference_label = f'gt_{REFERENCE_DISTANCE_KM}km'
df = df.dropna(subset=['spillover_ring'])
print('Untreated rows:', f'{len(df):,}')
print(df['spillover_ring'].value_counts().to_string())


## HT and Hajek ring contrasts

In [ ]:
spillover = ht_hajek_ring_diagnostics(
    df,
    outcome_col=OUTCOME_COL,
    year_col=YEAR_COL,
    ring_col='spillover_ring',
    reference_label=reference_label,
)
spillover_path = TABLE_DIR / append_tag_to_filename(SPILLOVER_TABLE_NAME, RUN_TAG)
spillover.to_csv(spillover_path, index=False)
print('Saved:', spillover_path)
print(spillover.head(12).to_string(index=False))


## Spatial HAC for Hajek ring contrasts

In [ ]:
spillover_hac = hac_for_hajek_ring_contrasts(
    df,
    centroids,
    cell_col=CELL_COL,
    year_col=YEAR_COL,
    outcome_col=OUTCOME_COL,
    ring_col='spillover_ring',
    reference_label=reference_label,
    cutoff_km=HAC_CUTOFF_KM,
    kernel=HAC_KERNEL,
    max_pairs=HAC_MAX_PAIRS,
)
spillover_hac_path = TABLE_DIR / append_tag_to_filename(SPILLOVER_HAC_TABLE_NAME, RUN_TAG)
spillover_hac.to_csv(spillover_hac_path, index=False)
print('Saved:', spillover_hac_path)
print(spillover_hac.head(12).to_string(index=False))
if not spillover_hac.empty and spillover_hac['hac_truncated'].any():
    print('Warning: at least one HAC calculation hit HAC_MAX_PAIRS. Increase HAC_MAX_PAIRS or reduce HAC_CUTOFF_KM after checking runtime/memory.')


## Event-time version

This version summarizes rings around the nearest treated cell by calendar year. It is not cohort-specific unless a nearest-treated-cohort lookup is added later.

In [ ]:
event_summary = (
    spillover.groupby('ring')
    .agg(
        mean_ht_difference=('ht_difference', 'mean'),
        mean_hajek_difference=('hajek_difference', 'mean'),
        p25_hajek_difference=('hajek_difference', lambda x: x.quantile(0.25)),
        p75_hajek_difference=('hajek_difference', lambda x: x.quantile(0.75)),
        n_years=('year', 'nunique'),
        mean_n_ring=('n_ring', 'mean'),
    )
    .reset_index()
)
event_path = TABLE_DIR / append_tag_to_filename(SPILLOVER_EVENT_TABLE_NAME, RUN_TAG)
event_summary.to_csv(event_path, index=False)
print('Saved:', event_path)
print(event_summary.to_string(index=False))


## Spillover profile figure

In [ ]:
fig_path = FIG_DIR / append_tag_to_filename(SPILLOVER_FIG_NAME, RUN_TAG)
if not spillover.empty:
    plot_df = spillover.pivot_table(index='year', columns='ring', values='hajek_difference', aggfunc='mean')
    ax = plot_df.plot(figsize=(11, 5), linewidth=1.5)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(f'Ring-based spillover contrasts relative to >{REFERENCE_DISTANCE_KM:g}km')
    ax.set_xlabel('Year')
    ax.set_ylabel('Hajek difference in annual forest loss (m2)')
    ax.grid(axis='y', alpha=0.3)
    ax.xaxis.grid(False)
    ax.legend(frameon=False, fontsize=9)
    fig = ax.get_figure()
    fig.tight_layout()
    fig.savefig(fig_path, dpi=220, bbox_inches='tight')
    plt.show()
    print('Saved:', fig_path)

print('Spatial HAC note: Hajek ring-contrast HAC estimates are saved in the dedicated HAC table. Full all-pairs matrices are never materialized; the KDTree routine stops if HAC_MAX_PAIRS is reached.')
